In [1]:
import sys
sys.path.append('../')
from datetime import datetime
import pytz, os
import numpy as np
import scipy as sp
from tqdm import tqdm
import scqubits as scq
import scqubits.settings as settings
import qutip as qt
from multiprocessing import Pool
from joblib import Parallel, delayed
import pandas as pd
import utils_2Q_gate_zp as ut
# Update scqubits settings
settings.OVERLAP_THRESHOLD = 0.3
max_step, nsteps = 1e-3, 1e4

In [2]:
folder = '../../data/3ncut_one_zeropi/'
evals = scq.read(folder + f'zeropi_0_specdata_truc=1000_3ncut.h5').energy_table
n_theta = scq.read(folder + f'zeropi_0_n_theta_truc=1000_3ncut.h5').matrixelem_table
n_phi = scq.read(folder + f'zeropi_0_n_phi_truc=1000_3ncut.h5').matrixelem_table
evals = evals - evals[0]

In [ ]:
# drive_phi, drive_theta, truc = False, True, 150
drive_phi, drive_theta, truc = True, False, 300
hspace_ntheta = [
0,   1,   2,   4,   5,   7,   8,  11,  12,  16,  17,  18,  21,
23,  25,  27,  30,  31,  32,  33,  37,  38,  40,  41,  42,  45,
46,  47,  48,  51,  52,  56,  59,  62,  63,  64,  65,  67,  72,
73,  74,  76,  78,  79,  82,  83,  85,  87,  88,  90,  92,  93,
96,  97,  98, 102, 103, 106, 107, 112, 114, 118, 119, 120, 121,
122, 123, 124, 127, 128, 131, 132, 133, 134, 137, 138, 142, 143
]
hspace_nphi = [
0,   2,   3,   4,   8,   9,  10,  11,  15,  16,  18,  19,  20,
23,  24,  25,  28,  31,  33,  35,  37,  39,  40,  42,  43,  46,
47,  49,  51,  53,  55,  56,  57,  60,  62,  64,  66,  67,  69,
70,  73,  76,  77,  79,  81,  83,  85,  86,  88,  91,  92,  95,
96,  98, 100, 101, 102, 104, 106, 108, 110, 112, 113, 116, 118,
120, 121, 123, 126, 128, 130, 132, 133, 136, 137, 139, 141, 143,
145, 146, 148, 151, 153, 154, 156, 158, 159, 162, 164, 166, 167,
169, 170, 173, 174, 176, 179, 181, 183, 185, 186, 188, 190, 192,
194, 196, 198, 200, 202, 205, 206, 208, 209, 212, 213, 216, 218,
220, 222, 223, 226, 227, 229, 230, 232, 234, 237, 239, 241, 242,
245, 246, 248, 250, 252, 253, 255, 257, 259, 262, 264, 265, 267,
269, 271, 273, 275, 277, 279, 281, 282, 283, 286, 288, 290, 291,
293, 294, 297, 299
]
hspace = hspace_ntheta if drive_theta else hspace_nphi
idx_2 = hspace.index(2)
hspace_len = len(hspace)

t1_other = 50 # μs
gamma_decay_logi =  1 / 1600e3
gamma_dephase_logi = 1 / 100e3
gamma_decay_other =  1 / 1e3 / t1_other
gamma_dephase_other = 1 / 1e3 / t1_other

if drive_theta:
    gamma_decay   = [0, gamma_decay_other,  gamma_decay_logi]  + [gamma_decay_other]  * (hspace_len-3)
    gamma_dephase = [0, gamma_dephase_other, gamma_dephase_logi] + [gamma_dephase_other] * (hspace_len-3)
else:
    gamma_decay   = [0,  gamma_decay_logi]  + [gamma_decay_other]  * (hspace_len-2)
    gamma_dephase = [0,  gamma_dephase_logi] + [gamma_dephase_other] * (hspace_len-2)

### $T_1$

$\gamma_{ll'} = \Gamma | \bra{l} n_\theta \ket{l'} |^2 = 1/ T_1  $ ( $T_1$ is intrawell decay time )

$\gamma_{07} = \Gamma_{07} | \bra{0} n_\theta \ket{7} |^2 = 1 / (2\times 10^{-6})   $

In [41]:
gamma_decay_0 = []
gamma_decay_2 = []
if drive_theta:
    Gamma_0 = gamma_decay_other / (n_theta[0,7]**2)
    Gamma_2 = gamma_decay_other / (n_theta[2,7]**2)
    for i in hspace:
        gamma_decay_0.append(Gamma_0* n_theta[0, i]**2)
        gamma_decay_2.append(Gamma_2* n_theta[0, i]**2)
else:
    Gamma_0 = gamma_decay_other / (n_phi[0,9]**2)
    Gamma_2 = gamma_decay_other / (n_phi[2,9]**2)
    for i in hspace:
        gamma_decay_0.append(Gamma_0* n_phi[0, i]**2)
        gamma_decay_2.append(Gamma_2* n_phi[0, i]**2)
gamma_decay_0[idx_2] = gamma_decay_logi
gamma_decay_2[idx_2] = gamma_decay_logi
print('np.imag(gamma_decay_0)=0 -->', np.all(np.imag(gamma_decay_0)==0))
print('np.imag(gamma_decay_2)=0 -->', np.all(np.imag(gamma_decay_2)==0))

if np.all(np.imag(gamma_decay_0)==0) and np.all(np.imag(gamma_decay_2)==0):
    gamma_decay_0 = np.real(gamma_decay_0)
    gamma_decay_2 = np.real(gamma_decay_2)
print('t1_other = %d μs'%t1_other)
print('gamma_decay_0=', gamma_decay_0.tolist())
print('gamma_decay_2=', gamma_decay_2.tolist())


np.imag(gamma_decay_0)=0 --> True
np.imag(gamma_decay_2)=0 --> True
t1_other = 50 μs
gamma_decay_0= [9.24878833313939e-35, 6.25e-07, 3.2239785055138662e-09, 3.3321952122994312e-34, 1.3411148909397326e-31, 2e-05, 0.003979646377759893, 4.5202598932261744e-30, 0.00016688438339287914, 3.594659646647095e-31, 5.504119894166018e-33, 8.515757692986305e-07, 4.49727918178021e-05, 6.423085139537547e-31, 1.1467809705056655e-06, 4.129391549948097e-32, 9.839376917210637e-07, 1.6032088897111164e-32, 6.203682171812002e-33, 2.036812279007242e-06, 1.926404244139375e-34, 6.689476042305062e-07, 2.4660713179099276e-32, 1.095794155241281e-32, 3.092905309817979e-07, 2.3194498678504048e-33, 5.4742318259733764e-33, 2.640773869802032e-08, 5.925715850237472e-32, 8.626704686548277e-06, 4.250661693660793e-06, 1.415635672896945e-32, 9.570397211583226e-07, 4.003973371462972e-07, 3.3142573340151597e-31, 1.5311803466866893e-34, 8.919515609567524e-06, 1.5542956781036325e-30, 8.153257700099807e-06, 2.9524973722777175e-0

In [ ]:
folder_1 = 'data/data_xgate_gamma_'
folder_2 = f'theta.txt' if drive_theta else f'phi.txt'
gamma_new = pd.read_csv(folder_1 + folder_2)
# gamma_decay_07 = gamma_new['gamma_decay_07'].to_numpy()
# gamma_decay_27 = gamma_new['gamma_decay_27'].to_numpy()
# gamma_decay_09 = gamma_new['gamma_decay_09'].to_numpy()
# gamma_decay_29 = gamma_new['gamma_decay_29'].to_numpy()
# hspace_ntheta = gamma_new['hspace_ntheta'].to_numpy()
# hspace_nphi = gamma_new['hspace_nphi'].to_numpy()
# g = np.column_stack((hspace_ntheta, hspace_nphi, gamma_decay_07, gamma_decay_27, gamma_decay_09, gamma_decay_29))
# df = pd.DataFrame(g, columns=['hspace_ntheta', 'hspace_nphi' ,'gamma_decay_07', 'gamma_decay_27','gamma_decay_09', 'gamma_decay_29',])
# gamma_new['gamma_decay_old'] = gamma_decay
# gamma_new['gamma_dephase_old'] = gamma_dephase
# gamma_new.to_csv(folder_1 + folder_2 , index=False)

In [38]:
desired_order = ['hspace', 'gamma_dephase_old', 'derivative', 'gamma_dephase',
                 'decay_2us_old', 'decay_2us_0', 'decay_2us_2', ]
gamma_new = gamma_new.reindex(columns=desired_order)
gamma_new

,hspace,gamma_dephase_old,derivative,gamma_dephase,decay_2us_old,decay_2us_0,decay_2us_2
0,0.0,0.000000,0.000000,0.000000e+00,0.000000e+00,2.312197e-33,1.414726e-32
1,2.0,0.000111,1.816109,1.111111e-04,6.250000e-07,6.250000e-07,6.250000e-07
2,3.0,0.000500,1.816056,7.988605e-06,5.000000e-04,8.059946e-08,4.931508e-07
3,4.0,0.000500,0.000271,1.190838e-09,5.000000e-04,8.330488e-33,5.097040e-32
4,8.0,0.000500,0.110984,4.882067e-07,5.000000e-04,3.352787e-30,2.051415e-29
...,...,...,...,...,...,...,...
155,291.0,0.000500,0.202705,8.916755e-07,5.000000e-04,1.889316e-10,1.155985e-09
156,293.0,0.000500,0.016108,7.085858e-08,5.000000e-04,3.089318e-10,1.890211e-09
157,294.0,0.000500,0.259940,1.143443e-06,5.000000e-04,7.534756e-35,4.610169e-34
158,297.0,0.000500,0.016562,7.285540e-08,5.000000e-04,6.038667e-37,3.694781e-36


In [42]:
gamma_new['decay_50us_old'] = gamma_decay
gamma_new['decay_50us_0'] = gamma_decay_0
gamma_new['decay_50us_2'] = gamma_decay_2
gamma_new

,hspace,gamma_dephase_old,derivative,gamma_dephase,decay_2us_old,decay_2us_0,decay_2us_2,decay_5us_old,decay_5us_0,decay_5us_2,decay_50us_old,decay_50us_0,decay_50us_2
0,0.0,0.000000,0.000000,0.000000e+00,0.000000e+00,2.312197e-33,1.414726e-32,0.000000e+00,9.248788e-34,5.658906e-33,0.000000e+00,9.248788e-35,5.658906e-34
1,2.0,0.000111,1.816109,1.111111e-04,6.250000e-07,6.250000e-07,6.250000e-07,6.250000e-07,6.250000e-07,6.250000e-07,6.250000e-07,6.250000e-07,6.250000e-07
2,3.0,0.000500,1.816056,7.988605e-06,5.000000e-04,8.059946e-08,4.931508e-07,2.000000e-04,3.223979e-08,1.972603e-07,2.000000e-05,3.223979e-09,1.972603e-08
3,4.0,0.000500,0.000271,1.190838e-09,5.000000e-04,8.330488e-33,5.097040e-32,2.000000e-04,3.332195e-33,2.038816e-32,2.000000e-05,3.332195e-34,2.038816e-33
4,8.0,0.000500,0.110984,4.882067e-07,5.000000e-04,3.352787e-30,2.051415e-29,2.000000e-04,1.341115e-30,8.205662e-30,2.000000e-05,1.341115e-31,8.205662e-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,291.0,0.000500,0.202705,8.916755e-07,5.000000e-04,1.889316e-10,1.155985e-09,2.000000e-04,7.557265e-11,4.623941e-10,2.000000e-05,7.557265e-12,4.623941e-11
156,293.0,0.000500,0.016108,7.085858e-08,5.000000e-04,3.089318e-10,1.890211e-09,2.000000e-04,1.235727e-10,7.560842e-10,2.000000e-05,1.235727e-11,7.560842e-11
157,294.0,0.000500,0.259940,1.143443e-06,5.000000e-04,7.534756e-35,4.610169e-34,2.000000e-04,3.013902e-35,1.844067e-34,2.000000e-05,3.013902e-36,1.844067e-35
158,297.0,0.000500,0.016562,7.285540e-08,5.000000e-04,6.038667e-37,3.694781e-36,2.000000e-04,2.415467e-37,1.477912e-36,2.000000e-05,2.415467e-38,1.477912e-37


### $T_\phi$

In [ ]:
def zeropi_eval(flux=0, truncation=300):
    ncut, phi_cut = 90, 300
    # Define system parameters (in GHz)
    EL = 0.377  # Inductive energy
    EJ = 6.013  # Josephson energy
    EC_phi = 1.142  # Phi mode charging energy
    EC_theta = 0.092  # Theta mode charging energy

    # Compute derived parameters
    E_CJ = 2 * EC_phi
    E_C = 2 / (1 / EC_theta - 1 / EC_phi)

    # Create the grid for the phi coordinate
    phi_grid = scq.Grid1d(-6 * np.pi, 6 * np.pi, phi_cut)

    # Initialize the Zero-Pi qubit system
    zero_pi = scq.ZeroPi(
        grid=phi_grid,
        EJ=EJ,
        EL=EL,
        ECJ=E_CJ,
        EC=E_C,
        dEJ=0.0,
        ng=0.0,
        flux=flux,
        ncut=ncut,
        truncated_dim=truncation,
    )

    # Compute the eigenvalues and construct the Hamiltonian
    evals, _ = zero_pi.eigensys(evals_count=truncation)
    evals = np.sort(evals)
    evals = evals - evals[0]
    return evals

eval_a = zeropi_eval(flux=0)
eval_b = zeropi_eval(flux=0.001)
derivative = [0,]
for i in hspace:
    derivative.append(np.abs(eval_b[i] - eval_a[i]) / 0.001)

In [ ]:
derivative = derivative_theta if drive_theta else derivative_phi
A = 1e-6
w_ir = 1e-9* 2 * np.pi # 1 Hz
t = 10e3 # 10 us = 10e3 ns
gamma_dephase_new = A * np.array(derivative) * np.sqrt(2 * np.abs(np.log(w_ir * t)))
gamma_dephase_new[idx_2] = gamma_dephase_logi
gamma_dephase_new

array([0.00000000e+00, 1.11111111e-04, 7.98860457e-06, 1.19083757e-09,
       4.88206675e-07, 4.56849571e-07, 1.02940459e-08, 2.43799741e-08,
       3.36017969e-08, 2.06765541e-07, 2.69821664e-07, 4.32170170e-07,
       3.18315442e-08, 1.24500583e-05, 1.18087993e-05, 6.08131004e-07,
       3.45262659e-09, 1.32148964e-07, 2.60443126e-08, 8.26769961e-08,
       7.16073802e-07, 2.83246220e-07, 3.31461748e-07, 1.02407867e-08,
       1.87397980e-07, 1.21509913e-07, 1.06525031e-07, 2.35727054e-07,
       2.28208326e-07, 1.30492024e-07, 1.06382933e-07, 4.72534726e-08,
       1.70183660e-07, 1.68907678e-06, 1.51996176e-06, 9.53603980e-08,
       1.20415192e-06, 1.18529837e-06, 9.36671248e-08, 4.56994501e-07,
       4.52585940e-07, 9.99856509e-09, 2.12040625e-07, 4.25521201e-08,
       2.45600444e-09, 1.43790503e-07, 4.40131120e-08, 9.76520418e-08,
       3.68869566e-07, 6.70483818e-08, 9.75980650e-08, 2.38980348e-07,
       3.24469834e-08, 6.36717776e-08, 4.18848750e-08, 1.15782347e-08,
      

In [ ]:
np.array(gamma_dephase)

array([0.        , 0.0005    , 0.00011111, 0.0005    , 0.0005    ,
       0.0005    , 0.0005    , 0.0005    , 0.0005    , 0.0005    ,
       0.0005    , 0.0005    , 0.0005    , 0.0005    , 0.0005    ,
       0.0005    , 0.0005    , 0.0005    , 0.0005    , 0.0005    ,
       0.0005    , 0.0005    , 0.0005    , 0.0005    , 0.0005    ,
       0.0005    , 0.0005    , 0.0005    , 0.0005    , 0.0005    ,
       0.0005    , 0.0005    , 0.0005    , 0.0005    , 0.0005    ,
       0.0005    , 0.0005    , 0.0005    , 0.0005    , 0.0005    ,
       0.0005    , 0.0005    , 0.0005    , 0.0005    , 0.0005    ,
       0.0005    , 0.0005    , 0.0005    , 0.0005    , 0.0005    ,
       0.0005    , 0.0005    , 0.0005    , 0.0005    , 0.0005    ,
       0.0005    , 0.0005    , 0.0005    , 0.0005    , 0.0005    ,
       0.0005    , 0.0005    , 0.0005    , 0.0005    , 0.0005    ,
       0.0005    , 0.0005    , 0.0005    , 0.0005    , 0.0005    ,
       0.0005    , 0.0005    , 0.0005    , 0.0005    , 0.0005 